# Jyotish LoRA: DPO training

Merges the SFT adapter into the fp16 base (that merged model is the reference), then trains a fresh LoRA with DPO on the preference pairs.

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------- settings ----------
SMOKE_TEST = True          # True: 6 steps on a handful of pairs, to check that it runs and fits
RUN_NAME = "astro-dpo-v2-4b"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SFT_HINT = ""              # substring of the SFT adapter path to pick when several inputs carry adapters
MAX_LENGTH = 2048          # pairs longer than this (prompt + answer) are dropped
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
BETA, LR, EPOCHS, SEED = 0.1, 2e-5, 2, 42
OUT_DIR = "/kaggle/working/" + RUN_NAME + ("-smoke" if SMOKE_TEST else "")

In [ ]:
import dataclasses, glob, json, math, time
import torch, transformers, peft, trl, datasets
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import DPOConfig, DPOTrainer

VERSIONS = {m.__name__: m.__version__ for m in (torch, transformers, peft, trl, datasets)}
print(VERSIONS)

def find(name):
    hits = glob.glob("/kaggle/input/**/" + name, recursive=True)
    assert hits, name + " not found"
    return hits[0]

adapters = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
adapters = [a for a in adapters if SFT_HINT in a] or adapters
SFT_ADAPTER = os.path.dirname(adapters[-1])
print("SFT adapter:", SFT_ADAPTER)

In [ ]:
used = torch.cuda.memory_allocated(0) / 2**30
assert used < 1, "GPU already in use: stop the session and run again"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(model, SFT_ADAPTER).merge_and_unload()   # reference = this merged SFT model
model.config.use_cache = False
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
for _p in model.parameters():
    if _p.requires_grad:
        _p.data = _p.data.float()
model.print_trainable_parameters()

In [ ]:
ds = load_dataset("json", data_files={"train": find("pairs.jsonl")})["train"]
ds = ds.map(lambda ex: {"prompt": ex["prompt"], "chosen": [{"role": "assistant", "content": ex["chosen"]}],
                        "rejected": [{"role": "assistant", "content": ex["rejected"]}]},
            remove_columns=ds.column_names)
def too_long(ex):
    n = max(len(tokenizer.apply_chat_template(ex["prompt"] + ex[k], tokenize=True, return_dict=True)["input_ids"])
            for k in ("chosen", "rejected"))
    return n > MAX_LENGTH
keep = [i for i, ex in enumerate(ds) if not too_long(ex)]
print(len(ds), "pairs;", len(ds) - len(keep), "dropped as longer than", MAX_LENGTH)
ds = ds.select(keep)
if SMOKE_TEST:
    ds = ds.select(range(min(48, len(ds))))

In [ ]:
total_steps = 6 if SMOKE_TEST else math.ceil(len(ds) / 8) * EPOCHS
config = dict(
    output_dir=OUT_DIR, beta=BETA, max_length=MAX_LENGTH, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    gradient_checkpointing=True, num_train_epochs=EPOCHS, max_steps=6 if SMOKE_TEST else -1, learning_rate=LR,
    lr_scheduler_type="cosine", warmup_steps=max(1, round(0.1 * total_steps)), fp16=True, bf16=False,
    logging_steps=2 if SMOKE_TEST else 5, save_strategy="no", seed=SEED, report_to="none",
    loss_type="sigmoid", precompute_ref_log_probs=False,
)
known = {f.name for f in dataclasses.fields(DPOConfig)}
print("NOTE: skipping settings this version does not accept:", sorted(set(config) - known))
args = DPOConfig(**{k: v for k, v in config.items() if k in known})
trainer = DPOTrainer(model=model, args=args, train_dataset=ds, processing_class=tokenizer)
from collections import Counter as _C
for _p in trainer.model.parameters():
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.float()
print("trainable dtypes:", dict(_C(str(_p.dtype) for _p in trainer.model.parameters() if _p.requires_grad)))

In [ ]:
t0 = time.time()
result = trainer.train()
hours = (time.time() - t0) / 3600
peak = torch.cuda.max_memory_reserved() / 2**30
print("done in %.2f h | peak GPU %.1f GiB of %.1f" % (hours, peak, torch.cuda.get_device_properties(0).total_memory / 2**30))
log = [h for h in trainer.state.log_history]
print([ {k: round(v, 4) for k, v in h.items() if k in ("loss", "rewards/accuracies", "rewards/margins")} for h in log if "loss" in h][-6:])
adapter_dir = OUT_DIR + "/adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
json.dump({"run": RUN_NAME, "smoke_test": SMOKE_TEST, "base_model": BASE_MODEL, "sft_adapter": SFT_ADAPTER,
           "dpo": {"beta": BETA, "lr": LR, "epochs": EPOCHS, "max_length": MAX_LENGTH, "pairs": len(ds),
                   "steps": result.global_step, "hours": round(hours, 2)},
           "lora": {"r": LORA_R, "alpha": LORA_ALPHA}, "peak_gpu_gib": round(peak, 1), "versions": VERSIONS,
           "log_history": log}, open(OUT_DIR + "/run_manifest.json", "w"), indent=2, default=str)